# SpaceX Historical Launch SQL Analysis

Queries calculated from the 90-record IBM course dataset, `dataset_part_2.csv`. Customer and detailed booster version fields are absent, so the NASA CRS and F9 v1.1 questions are **not answered** from this dataset.

In [13]:
import pandas as pd, sqlite3
df = pd.read_csv("dataset_part_2.csv")
print(df.shape)
conn = sqlite3.connect(":memory:")
df.to_sql("launches", conn, index=False)

(90, 18)


90

## Slide 24: Unique launch site names

Three unique launch sites in the 90-launch historical data.

*Matches the launch-site field in this source.*

In [14]:
query = 'SELECT DISTINCT LaunchSite FROM launches ORDER BY LaunchSite'
result = pd.read_sql_query(query, conn)
display(result)

,LaunchSite
0,CCAFS SLC 40
1,KSC LC 39A
2,VAFB SLC 4E


## Slide 25: First five CCAFS launch records

Five launch records from Cape Canaveral.

*Direct historical-subset result.*

In [15]:
query = "SELECT FlightNumber, Date, LaunchSite, ROUND(PayloadMass,0) AS Payload_kg FROM launches WHERE LaunchSite LIKE 'CCA%' LIMIT 5"
result = pd.read_sql_query(query, conn)
display(result)

,FlightNumber,Date,LaunchSite,Payload_kg
0,1,2010-06-04,CCAFS SLC 40,6105.0
1,2,2012-05-22,CCAFS SLC 40,525.0
2,3,2013-03-01,CCAFS SLC 40,677.0
3,5,2013-12-03,CCAFS SLC 40,3170.0
4,6,2014-01-06,CCAFS SLC 40,3325.0


## Slide 26: Total payload mass - available records

Total payload mass across all 90 historical launches.

*NASA (CRS) customer field is absent; a NASA-only total cannot be verified.*

In [16]:
query = 'SELECT ROUND(SUM(PayloadMass),1) AS Total_Payload_kg FROM launches'
result = pd.read_sql_query(query, conn)
display(result)

,Total_Payload_kg
0,549446.3


## Slide 27: Average payload mass - available records

Average payload mass across all 90 historical launches.

*F9 v1.1 is not separately identified in this dataset.*

In [17]:
query = 'SELECT ROUND(AVG(PayloadMass),1) AS Average_Payload_kg FROM launches'
result = pd.read_sql_query(query, conn)
display(result)

,Average_Payload_kg
0,6105.0


## Slide 28: First successful ground landing

First successful return-to-launch-site landing in this dataset.

*RTLS is the dataset proxy for successful ground landing.*

In [18]:
query = "SELECT MIN(Date) AS First_Ground_Landing FROM launches WHERE Outcome='True RTLS'"
result = pd.read_sql_query(query, conn)
display(result)

,First_Ground_Landing
0,2015-12-22


## Slide 29: Drone-ship landings, 4-6 t

Successful autonomous-drone-ship landings with payload 4-6 tonnes.

*Outcome = True ASDS; historical-subset query.*

In [19]:
query = "SELECT FlightNumber, ROUND(PayloadMass) AS Payload_kg FROM launches WHERE Outcome='True ASDS' AND PayloadMass > 4000 AND PayloadMass < 6000"
result = pd.read_sql_query(query, conn)
display(result)

,FlightNumber,Payload_kg
0,21,4696.0
1,24,4600.0
2,29,5300.0
3,40,5200.0
4,55,5800.0
5,70,5000.0


## Slide 30: Successful / unsuccessful booster landings

60 successful and 30 unsuccessful booster landing outcomes.

*This is BOOSTER success; mission outcome field is not available.*

In [20]:
query = 'SELECT Class AS Booster_Success, COUNT(*) AS Launches FROM launches GROUP BY Class ORDER BY Class'
result = pd.read_sql_query(query, conn)
display(result)

,Booster_Success,Launches
0,0,30
1,1,60


## Slide 31: Maximum payload in the dataset

Maximum observed payload: 15,600 kg on a Falcon 9 launch.

*BoosterVersion has one value: Falcon 9; no variant-level comparison.*

In [21]:
query = 'SELECT BoosterVersion, ROUND(MAX(PayloadMass),1) AS Max_Payload_kg FROM launches'
result = pd.read_sql_query(query, conn)
display(result)

,BoosterVersion,Max_Payload_kg
0,Falcon 9,15600.0


## Slide 32: Failed ASDS landings in 2015

Two unsuccessful drone-ship landings in 2015.

*Historical-subset query.*

In [22]:
query = "SELECT Date, FlightNumber, Outcome FROM launches WHERE Outcome='False ASDS' AND Date BETWEEN '2015-01-01' AND '2015-12-31'"
result = pd.read_sql_query(query, conn)
display(result)

,Date,FlightNumber,Outcome
0,2015-01-10,12,False ASDS
1,2015-04-14,14,False ASDS


## Slide 33: Landing outcome ranking, 2010-2017

Historical outcome categories, ranked by launch count.

*Period filter matches the requested interval.*

In [23]:
query = "SELECT Outcome, COUNT(*) AS Count FROM launches WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Outcome ORDER BY Count DESC"
result = pd.read_sql_query(query, conn)
display(result)

,Outcome,Count
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2
